# EfficientNet-B3 — Concrete Crack Classification

**Group 9 | Anuraag Dasari**

This notebook runs all EfficientNet-B3 experiments:
- Experiment 1: Model training & evaluation (accuracy, precision, recall, F1, confusion matrix)
- Experiment 2: Effect of training data size (25%, 50%, 75%, 100%)
- Experiment 3: Learning rate tuning (0.01, 0.001, 0.0001)
- Experiment 4: Cross-dataset generalization (Kaggle ↔ SDNET2018)

# CONFIGURATION

Change parameters here

In [ ]:
import os
import sys
from datetime import datetime

# Add src to path so we can import shared modules
sys.path.insert(0, os.path.abspath('../src'))

CONFIG = {
    # ---- DATASET ----
    "train_dataset_dir": "../data/processed/concrete_crack_75_10_15/",
    "sdnet_dataset_dir": "../data/processed_sdnet/",

    # ---- TRAINING ----
    "epochs": 10,
    "batch_size": 16,
    "learning_rate": 0.0001,
    "img_size": 224,

    # ---- MODEL ----
    "model_name": "EfficientNet-B3",
    "pretrained": True,
    "num_classes": 2,

    # ---- OUTPUT ----
    "output_dir": "../outputs/efficientnet_experiments",

    # ---- SEED ----
    "seed": 42,
}

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
SAVE_DIR = os.path.join(CONFIG["output_dir"], RUN_ID)
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.join(SAVE_DIR, 'figures'), exist_ok=True)

print("Run ID:", RUN_ID)
print("Save directory:", SAVE_DIR)

# IMPORTS & DEVICE

In [ ]:
import random
import csv

import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, Subset
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from dataset import CrackDataset
from models.efficientnet import EfficientNetB3Classifier
from transforms import get_classification_transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

CLASS_NAMES = ["crack", "no_crack"]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])

# HELPER FUNCTIONS

In [ ]:
def get_loaders(data_dir, batch_size, img_size, train_fraction=1.0, augment=False):
    """Build train, val, and test DataLoaders from a processed data directory."""
    train_transform = get_classification_transforms(img_size, train=augment)
    val_transform   = get_classification_transforms(img_size, train=False)

    train_data = CrackDataset(os.path.join(data_dir, 'train'), task='classification',
                               image_size=img_size, transform=train_transform)
    val_data   = CrackDataset(os.path.join(data_dir, 'val'),   task='classification',
                               image_size=img_size, transform=val_transform)
    test_data  = CrackDataset(os.path.join(data_dir, 'test'),  task='classification',
                               image_size=img_size, transform=val_transform)

    if train_fraction < 1.0:
        n = int(len(train_data) * train_fraction)
        indices = random.sample(range(len(train_data)), n)
        train_data = Subset(train_data, indices)

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_data,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_data,  batch_size=batch_size, shuffle=False)

    print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")
    return train_loader, val_loader, test_loader


def evaluate_model(model, loader, criterion):
    """Run model on a DataLoader and return loss + metrics."""
    model.eval()
    total_loss, y_true, y_pred = 0, [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    avg_loss  = total_loss / len(loader)
    accuracy  = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='binary', zero_division=0)
    recall    = recall_score(y_true, y_pred, average='binary', zero_division=0)
    f1        = f1_score(y_true, y_pred, average='binary', zero_division=0)
    cm        = confusion_matrix(y_true, y_pred)
    report    = classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0)

    return avg_loss, accuracy, precision, recall, f1, cm, report


def train_and_evaluate(label, train_loader, val_loader, test_loader,
                        lr=0.0001, epochs=None):
    """Train EfficientNet-B3 and return history + test results."""
    if epochs is None:
        epochs = CONFIG["epochs"]

    model = EfficientNetB3Classifier(
        pretrained=CONFIG["pretrained"],
        num_classes=CONFIG["num_classes"]
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "val_loss": [], "val_accuracy": [], "val_f1": []}

    print(f"\n{'='*60}")
    print(f"Training: {label} | LR={lr} | Epochs={epochs}")
    print(f"{'='*60}")

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)
        val_loss, val_acc, _, _, val_f1, _, _ = evaluate_model(model, val_loader, criterion)

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)
        history["val_f1"].append(val_f1)

        print(f"  Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val Acc: {val_acc:.4f} | "
              f"Val F1: {val_f1:.4f}")

    # Final test evaluation
    test_loss, test_acc, test_prec, test_rec, test_f1, test_cm, test_report = \
        evaluate_model(model, test_loader, criterion)

    print(f"\nTest Results for '{label}':")
    print(f"  Accuracy:  {test_acc:.4f}")
    print(f"  Precision: {test_prec:.4f}")
    print(f"  Recall:    {test_rec:.4f}")
    print(f"  F1:        {test_f1:.4f}")
    print(f"\nClassification Report:\n{test_report}")

    return model, history, {
        "experiment": label, "lr": lr,
        "test_accuracy": test_acc, "test_precision": test_prec,
        "test_recall": test_rec, "test_f1": test_f1,
        "test_cm": test_cm, "report": test_report
    }


def plot_confusion_matrix(cm, title, save_path=None):
    cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()


def plot_training_curves(history, title, save_path=None):
    epochs = list(range(1, len(history["train_loss"]) + 1))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(epochs, history["train_loss"], marker='o', label="Train Loss")
    ax1.plot(epochs, history["val_loss"],   marker='o', label="Val Loss")
    ax1.set_title(f"Loss: {title}")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()

    ax2.plot(epochs, history["val_accuracy"], marker='o', label="Val Accuracy")
    ax2.plot(epochs, history["val_f1"],       marker='o', label="Val F1")
    ax2.set_title(f"Metrics: {title}")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Score")
    ax2.legend()

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()

print("Helper functions defined.")

# EXPERIMENT 1: Baseline Model Training

Train EfficientNet-B3 on full Kaggle dataset with default hyperparameters.
Evaluate: Accuracy, Precision, Recall, F1-score, Confusion Matrix.

In [ ]:
set_seed(CONFIG["seed"])

train_loader, val_loader, test_loader = get_loaders(
    data_dir=CONFIG["train_dataset_dir"],
    batch_size=CONFIG["batch_size"],
    img_size=CONFIG["img_size"],
    train_fraction=1.0,
    augment=False
)

baseline_model, baseline_history, baseline_result = train_and_evaluate(
    label="Baseline (100% data, LR=0.0001)",
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    lr=0.0001
)

In [ ]:
# Plot baseline training curves
plot_training_curves(
    baseline_history,
    title="EfficientNet-B3 Baseline",
    save_path=os.path.join(SAVE_DIR, 'figures', 'baseline_training_curves.png')
)

# Plot baseline confusion matrix
plot_confusion_matrix(
    baseline_result["test_cm"],
    title="Baseline Confusion Matrix",
    save_path=os.path.join(SAVE_DIR, 'figures', 'baseline_confusion_matrix.png')
)

# EXPERIMENT 2: Training Data Size

Train with 25%, 50%, 75%, and 100% of the training data.
Compare how data size affects performance.

In [ ]:
data_size_fractions = [0.25, 0.50, 0.75, 1.0]
data_size_results = []

for fraction in data_size_fractions:
    set_seed(CONFIG["seed"])
    label = f"Data Size {int(fraction * 100)}%"

    tr_loader, v_loader, te_loader = get_loaders(
        data_dir=CONFIG["train_dataset_dir"],
        batch_size=CONFIG["batch_size"],
        img_size=CONFIG["img_size"],
        train_fraction=fraction,
        augment=False
    )

    _, history, result = train_and_evaluate(
        label=label,
        train_loader=tr_loader,
        val_loader=v_loader,
        test_loader=te_loader,
        lr=0.0001
    )

    result["train_fraction"] = fraction
    data_size_results.append(result)

    plot_confusion_matrix(
        result["test_cm"],
        title=f"Confusion Matrix: {label}",
        save_path=os.path.join(SAVE_DIR, 'figures', f'cm_data_size_{int(fraction*100)}.png')
    )

In [ ]:
# Plot data size vs performance
fractions_pct = [r["train_fraction"] * 100 for r in data_size_results]
f1_scores     = [r["test_f1"]        for r in data_size_results]
accuracies    = [r["test_accuracy"]   for r in data_size_results]

plt.figure(figsize=(8, 5))
plt.plot(fractions_pct, f1_scores,  marker='o', label='Test F1')
plt.plot(fractions_pct, accuracies, marker='s', label='Test Accuracy')
plt.xlabel("Training Data Used (%)")
plt.ylabel("Score")
plt.title("EfficientNet-B3: Effect of Training Data Size")
plt.xticks(fractions_pct)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'figures', 'data_size_experiment.png'), dpi=150)
plt.show()

# Summary table
df_data_size = pd.DataFrame(data_size_results)[['experiment','train_fraction',
                                                  'test_accuracy','test_precision',
                                                  'test_recall','test_f1']]
print(df_data_size.to_string(index=False))

# EXPERIMENT 3: Learning Rate Tuning

Compare learning rates: 0.01, 0.001, 0.0001

In [ ]:
learning_rates = [0.01, 0.001, 0.0001]
lr_results = []

# Use full data for LR experiment
set_seed(CONFIG["seed"])
tr_loader, v_loader, te_loader = get_loaders(
    data_dir=CONFIG["train_dataset_dir"],
    batch_size=CONFIG["batch_size"],
    img_size=CONFIG["img_size"],
    train_fraction=1.0,
    augment=False
)

for lr in learning_rates:
    set_seed(CONFIG["seed"])
    label = f"LR={lr}"

    _, history, result = train_and_evaluate(
        label=label,
        train_loader=tr_loader,
        val_loader=v_loader,
        test_loader=te_loader,
        lr=lr
    )

    lr_results.append(result)

    plot_training_curves(
        history,
        title=f"EfficientNet-B3 {label}",
        save_path=os.path.join(SAVE_DIR, 'figures', f'training_curve_lr_{str(lr).replace(".","_")}.png')
    )
    plot_confusion_matrix(
        result["test_cm"],
        title=f"Confusion Matrix: {label}",
        save_path=os.path.join(SAVE_DIR, 'figures', f'cm_lr_{str(lr).replace(".","_")}.png')
    )

In [ ]:
# Learning rate comparison bar chart
lr_labels   = [r["experiment"] for r in lr_results]
lr_f1       = [r["test_f1"]    for r in lr_results]
lr_accuracy = [r["test_accuracy"] for r in lr_results]

x = range(len(lr_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([i - width/2 for i in x], lr_accuracy, width, label='Test Accuracy')
ax.bar([i + width/2 for i in x], lr_f1,       width, label='Test F1')
ax.set_xticks(list(x))
ax.set_xticklabels(lr_labels)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.set_title("EfficientNet-B3: Effect of Learning Rate")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'figures', 'lr_comparison.png'), dpi=150)
plt.show()

# Summary table
df_lr = pd.DataFrame(lr_results)[['experiment','test_accuracy','test_precision','test_recall','test_f1']]
print(df_lr.to_string(index=False))

# EXPERIMENT 4: Cross-Dataset Generalization

Train on Kaggle → Test on SDNET2018, and vice versa.
Compare in-domain vs cross-domain performance.

In [ ]:
cross_dataset_results = []

# -------------------------------------------------------
# Direction 1: Train on Kaggle, test on Kaggle and SDNET
# -------------------------------------------------------
set_seed(CONFIG["seed"])
kaggle_train, kaggle_val, kaggle_test = get_loaders(
    data_dir=CONFIG["train_dataset_dir"],
    batch_size=CONFIG["batch_size"],
    img_size=CONFIG["img_size"]
)
_, _, sdnet_test = get_loaders(
    data_dir=CONFIG["sdnet_dataset_dir"],
    batch_size=CONFIG["batch_size"],
    img_size=CONFIG["img_size"]
)

kaggle_model, _, kaggle_in_domain = train_and_evaluate(
    label="Kaggle → Kaggle (in-domain)",
    train_loader=kaggle_train,
    val_loader=kaggle_val,
    test_loader=kaggle_test,
    lr=0.0001
)
cross_dataset_results.append(kaggle_in_domain)

# Test trained Kaggle model on SDNET (cross-domain)
criterion = nn.CrossEntropyLoss()
_, cross_acc, cross_prec, cross_rec, cross_f1, cross_cm, cross_report = \
    evaluate_model(kaggle_model, sdnet_test, criterion)

print("\nKaggle → SDNET (cross-domain):")
print(f"  Accuracy: {cross_acc:.4f} | F1: {cross_f1:.4f}")
print(cross_report)

cross_dataset_results.append({
    "experiment": "Kaggle → SDNET (cross-domain)",
    "test_accuracy": cross_acc, "test_precision": cross_prec,
    "test_recall": cross_rec, "test_f1": cross_f1,
    "test_cm": cross_cm
})

plot_confusion_matrix(cross_cm, "Kaggle → SDNET (cross-domain)",
    save_path=os.path.join(SAVE_DIR, 'figures', 'cm_kaggle_to_sdnet.png'))

In [ ]:
# Summary of cross-dataset results so far
print("\nCross-dataset experiment complete.")
print(f"In-domain (Kaggle→Kaggle): Acc={kaggle_in_domain['test_accuracy']:.4f} | F1={kaggle_in_domain['test_f1']:.4f}")
print(f"Cross-domain (Kaggle→SDNET): Acc={cross_acc:.4f} | F1={cross_f1:.4f}")

In [ ]:
# Cross-dataset summary comparison chart
cd_names = [r["experiment"] for r in cross_dataset_results]
cd_acc   = [r["test_accuracy"] for r in cross_dataset_results]
cd_f1    = [r["test_f1"]       for r in cross_dataset_results]

x = range(len(cd_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i - width/2 for i in x], cd_acc, width, label='Test Accuracy')
ax.bar([i + width/2 for i in x], cd_f1,  width, label='Test F1')
ax.set_xticks(list(x))
ax.set_xticklabels(cd_names, rotation=15, ha='right')
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.set_title("EfficientNet-B3: Cross-Dataset Generalization")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'figures', 'cross_dataset_comparison.png'), dpi=150)
plt.show()

# Summary table
df_cross = pd.DataFrame(cross_dataset_results)[['experiment','test_accuracy',
                                                  'test_precision','test_recall','test_f1']]
print(df_cross.to_string(index=False))

# SAVE ALL RESULTS

In [ ]:
# Combine all experiment results (excluding cm and report objects)
all_results = []

for r in [baseline_result] + data_size_results + lr_results + cross_dataset_results:
    row = {k: v for k, v in r.items() if k not in ('test_cm', 'report')}
    all_results.append(row)

df_all = pd.DataFrame(all_results)
summary_path = os.path.join(SAVE_DIR, 'efficientnet_all_results.csv')
df_all.to_csv(summary_path, index=False)
print(f"✅ Saved all results to: {summary_path}")
print(df_all[['experiment','test_accuracy','test_precision','test_recall','test_f1']].to_string(index=False))

In [ ]:
# Append to shared experiments_classification/summary.csv
shared_summary_path = "../experiments_classification/summary.csv"

for r in all_results:
    r["model"] = "EfficientNet-B3"
    r["run_id"] = RUN_ID

if os.path.exists(shared_summary_path):
    df_old = pd.read_csv(shared_summary_path)
    df_new = pd.concat([df_old, pd.DataFrame(all_results)], ignore_index=True)
else:
    df_new = pd.DataFrame(all_results)

df_new.to_csv(shared_summary_path, index=False)
print(f"✅ Updated shared summary.csv at: {shared_summary_path}")